# Environment Deep Dive


## The Role of the Environment

Our environment serves as the simulation engine that creates the dynamic interactions between applicants, suppliers, and the university (main players).

It is designed to:
- Generate synthetic datasets representing diverse applicants (with different skills and grades)
- Model the university’s hidden criteria within different faculties (with different requirements).
- Represent how third-party suppliers in a meaningful way (so they can impact results)
- Provide a controlled yet flexible space for experimenting with different strategic scenarios.

The environment leverages vector-based representations of the key players, allowing for efficient calculations of outcomes.


## UniversityMLP

### Purpose
The model serves as university's decision-making - predicting *final grades* for each faculty given an applicant's feature vector.
### Main Components:
- Works with the number of features and faculties
- **Feed-Forward MLP** - outputs the predicted final grades for all faculties.

The Model learn the nonlinear relationships between features and success across faculties.


## ApplicantMLP

### Purpose
Allows applicants to learn and predict **which faculty they’re most likely to be assigned based on their (possibly manipulated) features**.
### Main Parameters and Components:
- Works with the number of features and faculties
- **Feed-Forward MLP** - outputs the predicted final grades for all faculties.
- **Softmax output** - Produces a probability distribution over faculties.

The Model is used by the applicant to quantify the impact of supplier modifications on the likelihood of achieving the desired outcome.



## Environment Params
### FacultyParams
- Represents the hidden success criteria of each faculty. 
- Each Faculty has *utility vectors* with weights indicating the importance of each feature gore the faculty.
- weights are normalize and randomly assigned
- **Goal** -  to be as realitic as possible while being diverse enough to create seperation between faculties

### SupplierParams
- Represents hird-party suppliers offering feature modifications to applicants.
- Each Supplier has *diff vectors* with (partial) modification weights for boosting or negating specific features.
- **Goal** - to mimic supplier behavior in real life while being effective enough to effect the decision making process.

## UniversityEnvironment ##
### Purpose ## 
Acts as the simulation manager — creating faculties, suppliers, applicants, and running the mechanics of the game.

### Main Design Ideas ###
- control the Number of applicant features, university faculties and third-party suppliers (including uni-supplier).
- Maneges the complexity (deciding how many features and noise are added) and size (how many applicants, suppliers and faculties are included) of the system.
- Mimicing real-world mechanisms to try to model realistic scenarios (focusing of critical features/scores per faculty, creating trade-offs for suppliers, adding noise between expectation and actual results) 

### DF Generations ###
- Creates historical data for training ML models (cold start data frames). These past applicants serves at the applicants for the university to use and for the applicants to learn from (mimic how future applicants learn by past student what it's best to focus on for getting to the desired faculty). **Goal** - Simulate a real-world dataset of past university decisions.
- Create current applicants DF (stratigic students). **Goal** - Provide a testbed for current strategies, manipulation, and experiments.
- all student are assigned with desired faculty randomly - for diverse faculty desires.

## Scenario Variations

After establishing the base scenario of **"Modified with Feature Knowledge"**, we explored several variations. Each variation models a different strategic scenario or changes in information dynamics, designed to test the robustness of the university system under varying degrees of manipulation and knowledge.

For each variation, we will show the core changes from the base scenario, the intuition behind it, and the key evaluation metrics. We will mainly focus on two Evaluation Metrics **total mean grades and percentage of students assigned to their desired faculty**.

### 1. Modified Without Feature Knowledge

#### Description:
In this scenario, applicants do **not** have access to their true feature vector. Instead, thier choice is soppused to represent desicions that rely on general knowledge or assumptions when choosing a supplier. Key Changes include applicants choosing suppliers **do not know their own feature vector**. In Additions, supplier impact is less predictable, possibly leading to mismatches or inefficient manipulation.

#### Expected Impact:
- **General sinilar or even lower percentage of students** achieving their desired faculty compared to the base. this based on the fact that Without knowledge of their own strengths/weaknesses, students cannot select the supplier that truly helps with their desired faculty. There is no proven way for them to improve their abilities.
- Potentially **lower mean grades**, as manipulation is less precise (even though the loss of mean grade will be small and accour only on the rare cases were students accually changed thier faculties, even if not to the desired one).

### Code Modifications:
- Applicants select suppliers **without access to their own features** by setting `use_feature_knowledge=False` during supplier selection. instead we use base vector (generic for all students) and try to find the modification that potentially can benefit the students (supplier that seems most effective)
- **Feature-blending logic was introduced**: When applicants lack knowledge of their own features, the supplier’s modification is blended carefully with the original features. Specifically, **only the modified feature indices** from the supplier are applied, while all unmodified features default to the applicant’s original values.


### 2. Fully Exposed Supplier Scenario

#### Description:
In this scenario, suppliers are given **full access to the university model**, allowing them to perfectly optimize their modifications based on exactly how the university predicts grades.
Suppliers simulate the effect of every possible modification and choose the one that **maximizes the university's predicted grade for the applicant's desired faculty**.

#### Expected Impact:
- We beilive this scenario will allow the hightest precentage of student to reach thier desired faculty because the suppliers now exploit the exact logic of the university model - which allow the students to know in advance if and how they can get their desired faculty.
- **Lower mean grades**, as suppliers are able to manipulate the university into making the most suboptimal assignments based on inflated or targeted feature changes.
- This scenario demonstrates the dangers of full system transparency in a strategic environment, where external agents can reverse-engineer the system and, as a result, lowering the system success too much.

#### Code Modifications:
- Supplier selection logic changes to **simulate the university model predictions for every supplier's modification**.
- Suppliers choose the modification that **maximizes the university’s predicted grade** for the student’s desired faculty using `choose_supplier_for_applicant_fully_exposed()`.

### 3. University Reconstruction Scenario

#### Description:
In this scenario, the university **actively tries to reverse supplier manipulation** by reconstructing the original applicant feature vectors before making faculty assignment decisions. 
The university uses group-based proportional adjustments based on applicant groups with the same desired faculty to estimate the unmodified features.

#### Expected Impact:
- **Higher mean grades** because the university is better able to match students based on their real abilities rather than manipulated features (even if this method will not achive the result of the original, no-gaming, option)
- **Reduced effectiveness of manipulation**: Many students may fail to get into their desired faculty because the university partially "undoes" the supplier’s impact (it can even effect the deserving students).
- This simulates a realistic defensive mechanism where institutions **learn patterns of manipulation** and correct for them - while not fully taking into account the effect of the reconstruction on all students.

#### Code Modifications:
- The faculty assignment function is changed to **use reconstructed features** via `assign_applicants_to_faculties_with_reconstruction()`.
- Before running the university model, the environment performs **proportion-based reconstruction** that neutralizes supplier manipulation.

### 4. University as a Supplier Scenario

#### Description:
This variation introduces the university as an **optional, protective supplier**. Applicants who cannot find a third-party supplier that significantly improves their chances **can choose the university** instead. 
The university supplier provides controlled, less risky modifications while also **emphasizing the importance of student fulfillment** (when possible) — trying to find a middle ground between maximizing university success (grades) and supporting student welfare.

#### Expected Impact:
- **Maxining fairness** by achiving most students getting to thier desired faculty (even in the expense of university success.) 
- **middle average grades**  - better then student manipulation option while not achiving the original university option. Students who are at risk of making bad supplier choices now have a safety net - that will not always apply.
- **More stable assignments** since the university intervenes to limit unnecessary or harmful manipulation.

#### Code Modifications:
- The supplier selection step now includes a **threshold check**: If no supplier improves the success probability beyond a desired level, the student **defaults to the university supplier**.
- The environment tracks which students **chose the university** as their supplier (`env.university_applicants`) and ensures these cases are handled in the assignment logic.
- During assignment, the `assign_applicants_to_faculties()` method includes a **special condition**: if the student is a university applicant and the predicted grade for their desired faculty is greater than or equal to 55, the university guarantees assigning them to their desired faculty.
